# Therapy Assistant OpenEnv Colab Run

This notebook runs the current environment-backed training flow:

1. install dependencies
2. generate future-reward data with `ESCEnv` rollouts
3. train the scalar reward model
4. run GRPO on the policy model


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = os.environ.get("THERAPY_ASSISTANT_REPO_URL", "")
REPO_DIR = Path("/content/meta-hackathon")

if not REPO_DIR.exists():
    if not REPO_URL:
        raise RuntimeError(
            "Clone the repo into /content/meta-hackathon first, or set THERAPY_ASSISTANT_REPO_URL before running this notebook."
        )
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")


In [ ]:
!pip install -r requirements-training.txt

In [ ]:
import os

os.environ.setdefault("API_BASE_URL", "https://router.huggingface.co/v1")

if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("Set HF_TOKEN in the Colab environment before running training.")

POLICY_MODEL = os.environ.get("POLICY_MODEL", "Qwen/Qwen2.5-3B-Instruct")
CRITIC_MODEL = os.environ.get("CRITIC_MODEL", "Qwen/Qwen2.5-7B-Instruct")
REWARD_MODEL = os.environ.get("REWARD_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
DATASET_NAME = os.environ.get("DATASET_NAME", "thu-coai/esconv")
MAX_SEED_EXAMPLES = int(os.environ.get("MAX_SEED_EXAMPLES", "200"))
ROLLOUT_STEPS = int(os.environ.get("ROLLOUT_STEPS", "6"))
MAX_TURNS = int(os.environ.get("MAX_TURNS", "16"))

print({
    "API_BASE_URL": os.environ["API_BASE_URL"],
    "POLICY_MODEL": POLICY_MODEL,
    "CRITIC_MODEL": CRITIC_MODEL,
    "REWARD_MODEL": REWARD_MODEL,
    "DATASET_NAME": DATASET_NAME,
    "MAX_SEED_EXAMPLES": MAX_SEED_EXAMPLES,
    "ROLLOUT_STEPS": ROLLOUT_STEPS,
    "MAX_TURNS": MAX_TURNS,
})


In [ ]:
!python -m training.simulate_dialogues \
  --output-dir artifacts/sim_data \
  --examples-source esconv_hf \
  --dataset-name "$DATASET_NAME" \
  --dataset-split train \
  --max-seed-examples "$MAX_SEED_EXAMPLES" \
  --policy-model "$POLICY_MODEL" \
  --critic-model "$CRITIC_MODEL" \
  --episodes-per-seed 2 \
  --num-candidates 4 \
  --rollout-steps "$ROLLOUT_STEPS" \
  --max-turns "$MAX_TURNS"

In [ ]:
!python -m training.reward_model \
  --input-jsonl artifacts/sim_data/candidate_rewards.jsonl \
  --model-name "$REWARD_MODEL" \
  --output-dir artifacts/reward_model

In [ ]:
!accelerate launch -m training.grpo_policy \
  --prompt-jsonl artifacts/sim_data/candidate_rewards.jsonl \
  --model-name "$POLICY_MODEL" \
  --reward-model-dir artifacts/reward_model \
  --output-dir artifacts/grpo_policy

In [ ]:
!find artifacts -maxdepth 2 -type f | sort